# Animacion de descenso por gradiente

Este notebook genera una animacion del descenso por gradiente sobre la funcion de Rosenbrock en 2D y guarda el resultado en la carpeta `datos`.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation

from funciones_gradientes import run_gradient_descent, rosenbrock_gradient
from funciones_objetivo import rosenbrock

In [ ]:
DATOS_DIR = Path("datos")
DATOS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "initial_position": np.array([-1.7, 1.8], dtype=float),
    "rate": 0.001,
    "max_iterations": 250,
    "tolerance": 1e-8,
    "x_limits": (-2.0, 2.0),
    "y_limits": (-1.0, 3.0),
    "grid_points": 250,
    "gif_name": "animacion_gradiente_rosenbrock_2d.gif",
}

In [ ]:
resultado = run_gradient_descent(
    initial_position=CONFIG["initial_position"],
    function=rosenbrock,
    gradient_function=rosenbrock_gradient,
    rate=CONFIG["rate"],
    max_iterations=CONFIG["max_iterations"],
    tolerance=CONFIG["tolerance"],
)

trayectoria = resultado["history"]

print("Posicion inicial:", resultado["initial_position"])
print("Posicion final:", resultado["final_position"])
print("Valor final:", resultado["final_value"])
print("Iteraciones:", resultado["iterations"])

In [ ]:
x = np.linspace(*CONFIG["x_limits"], CONFIG["grid_points"])
y = np.linspace(*CONFIG["y_limits"], CONFIG["grid_points"])
X, Y = np.meshgrid(x, y)
Z = np.array([rosenbrock(np.array([x1, x2], dtype=float)) for x1, x2 in zip(X.ravel(), Y.ravel())])
Z = Z.reshape(X.shape)

plt.figure(figsize=(8, 6))
niveles = np.logspace(-1, 3.5, 25)
plt.contour(X, Y, Z, levels=niveles, norm="log", cmap="viridis")
plt.plot(trayectoria[:, 0], trayectoria[:, 1], color="crimson", linewidth=2)
plt.scatter(trayectoria[0, 0], trayectoria[0, 1], color="black", label="Inicio")
plt.scatter(trayectoria[-1, 0], trayectoria[-1, 1], color="gold", edgecolor="black", label="Final")
plt.title("Trayectoria del descenso por gradiente")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
niveles = np.logspace(-1, 3.5, 25)
ax.contour(X, Y, Z, levels=niveles, norm="log", cmap="viridis")
ax.set_title("Descenso por gradiente sobre Rosenbrock 2D")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_xlim(CONFIG["x_limits"])
ax.set_ylim(CONFIG["y_limits"])

linea, = ax.plot([], [], color="crimson", linewidth=2)
punto, = ax.plot([], [], marker="o", color="black")
texto = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")

def init():
    linea.set_data([], [])
    punto.set_data([], [])
    texto.set_text("")
    return linea, punto, texto

def update(frame):
    xs = trayectoria[: frame + 1, 0]
    ys = trayectoria[: frame + 1, 1]
    linea.set_data(xs, ys)
    punto.set_data([trayectoria[frame, 0]], [trayectoria[frame, 1]])
    valor = rosenbrock(trayectoria[frame])
    texto.set_text(f"Paso: {frame} | f(x): {valor:.6f}")
    return linea, punto, texto

animacion = animation.FuncAnimation(
    fig,
    update,
    frames=len(trayectoria),
    init_func=init,
    interval=80,
    blit=True,
)

plt.close(fig)
HTML(animacion.to_jshtml())

In [ ]:
gif_path = DATOS_DIR / CONFIG["gif_name"]
animacion.save(gif_path, writer=animation.PillowWriter(fps=12))
print(f"Animacion guardada en: {gif_path.resolve()}")